# Week 5 - Variant Analysis of CYP2C8, CYP2C9, and CYP2C19 (hg38)

In this notebook, we will build a small bioinformatics pipeline to analyze three important **drug metabolism genes** on **chromosome 10**:  
**CYP2C8**, **CYP2C9**, and **CYP2C19**.  

Here, we use two types of sequencing data:

- **Illumina:** short, accurate reads  
- **PacBio:** long, less accurate reads  

**Using UCSC Genome Browser, we can find that they are located in the following positions within the hg38 version of the human genome:**

- **CYP2C8:** chr10:95036772-95069497
- **CYP2C9:** chr10:94938658-94990091
- **CYP2C19:** chr10:94762681-94855547



## Step 1: Download chr10 (hg38) genome, then download and unpack the FASTQ files

In [ ]:
%%bash

set -euo pipefail

if [ ! -f chr10.fa ]; then
  echo "[week5] downloading hg38 chr10..."
  curl -L -o chr10.fa.gz https://hgdownload.cse.ucsc.edu/goldenpath/hg38/chromosomes/chr10.fa.gz
  gunzip -c chr10.fa.gz > chr10.fa
  rm -f chr10.fa.gz
else
  echo "[week5] chr10.fa already present — skipping download."
fi


In [ ]:
%%bash
set -euo pipefail

# make sure the folder exists
mkdir -p data

# Download (Illumina + PacBio)
curl -L https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/illumina.fq.bz2 -o data/illumina.fq.bz2
curl -L https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/pacbio.fq.bz2   -o data/pacbio.fq.bz2

# Uncompress to .fq 
bunzip2 -kf data/illumina.fq.bz2
bunzip2 -kf data/pacbio.fq.bz2 

## Step 2: Align all samples in FASTQ format to the human genome (version GRCh38)

In [ ]:
%%bash

# Aligning using minimap
minimap2 -a chr10.fa data/illumina.fq | samtools view -bS | samtools sort -o illumina.sorted.bam
samtools index illumina.sorted.bam

# Using samtools to get bam and bai files
minimap2 -a chr10.fa data/pacbio.fq | samtools view -bS | samtools sort -o pacbio.sorted.bam
samtools index pacbio.sorted.bam

## Step 3: Finding all the variants

In [ ]:
%%bash

bcftools mpileup -f chr10.fa -r 'chr10:95036772-95069497','chr10:94938658-94990091','chr10:94762681-94855547' illumina.sorted.bam | bcftools call -mv -Ov -o illumina.vcf
bcftools mpileup -f chr10.fa -r 'chr10:95036772-95069497','chr10:94938658-94990091','chr10:94762681-94855547' pacbio.sorted.bam | bcftools call -mv -Ov -o pacbio.vcf

## Step 4: Phasing the variants with HapCUT2  

In [ ]:
%%bash
set -euo pipefail

# --- Phasing 
extractHAIRS --bam illumina.sorted.bam --VCF illumina.vcf --ref chr10.fa --out illumina.frag
hapcut2      --fragments illumina.frag  --VCF illumina.vcf  --output illumina.hapcut
extractHAIRS --bam pacbio.sorted.bam   --VCF pacbio.vcf   --ref chr10.fa --out pacbio.frag
hapcut2      --fragments pacbio.frag   --VCF pacbio.vcf   --output pacbio.hapcut
rm -f illumina.frag pacbio.frag  # keep the VCFs

# --- Normalize names to the ones we’ll use downstream ---
mv -f illumina.hapcut.phased.VCF illumina.phased.vcf
mv -f pacbio.hapcut.phased.VCF   pacbio.phased.vcf

# --- Compress & index (sequential; no races) ---
bgzip -f illumina.phased.vcf
bgzip -f pacbio.phased.vcf
tabix -f -p vcf illumina.phased.vcf.gz
tabix -f -p vcf pacbio.phased.vcf.gz

# --- Compare: shared vs unique ---
rm -rf variants
bcftools isec -p variants illumina.phased.vcf.gz pacbio.phased.vcf.gz


## Step 5: Comparing the variants

In [ ]:
%%bash
set -euo pipefail

ILL=illumina.phased.vcf.gz
PAC=pacbio.phased.vcf.gz
[ -s "$ILL" ] && [ -s "$PAC" ] || { echo "Missing phased VCFs (.gz)."; exit 1; }

# Regions (hg38)
C19="chr10:94762681-94855547"  # CYP2C19
C9="chr10:94938658-94990091"   # CYP2C9
C8="chr10:95036772-95069497"   # CYP2C8

count_vcf () { [ -f "$1" ] && grep -vc '^#' "$1" || echo 0; }

# Overall isec
rm -rf variants_overall
bcftools isec -p variants_overall "$ILL" "$PAC" >/dev/null
S_all=$(count_vcf variants_overall/0002.vcf)   # shared
I_all=$(count_vcf variants_overall/0000.vcf)   # illumina-only
P_all=$(count_vcf variants_overall/0001.vcf)   # pacbio-only
T_all=$((S_all+I_all+P_all))

# Per-gene isec
rm -rf isec_c19 isec_c9 isec_c8
bcftools isec -r "$C19" -p isec_c19 "$ILL" "$PAC" >/dev/null
bcftools isec -r "$C9"  -p isec_c9  "$ILL" "$PAC" >/dev/null
bcftools isec -r "$C8"  -p isec_c8  "$ILL" "$PAC" >/dev/null

S_c19=$(count_vcf isec_c19/0002.vcf); I_c19=$(count_vcf isec_c19/0000.vcf); P_c19=$(count_vcf isec_c19/0001.vcf); T_c19=$((S_c19+I_c19+P_c19))
S_c9=$(count_vcf  isec_c9/0002.vcf);  I_c9=$(count_vcf  isec_c9/0000.vcf);  P_c9=$(count_vcf  isec_c9/0001.vcf);  T_c9=$((S_c9+I_c9+P_c9))
S_c8=$(count_vcf  isec_c8/0002.vcf);  I_c8=$(count_vcf  isec_c8/0000.vcf);  P_c8=$(count_vcf  isec_c8/0001.vcf);  T_c8=$((S_c8+I_c8+P_c8))

# Write TSV
cat > variant_summary.tsv <<EOF
    	Shared	Illumina_only	PacBio_only	Total
Overall	$S_all	$I_all	$P_all	$T_all
CYP2C19	$S_c19	$I_c19	$P_c19	$T_c19
CYP2C9	$S_c9	$I_c9	$P_c9	$T_c9
CYP2C8	$S_c8	$I_c8	$P_c8	$T_c8
EOF

echo "Variant comparison summary:"
echo " "
column -t -s $'\t' variant_summary.tsv

